# NB03: Data Analysis

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250104200

</div>

## Setup
Import packages

In [22]:
import pandas as pd
import plotly.express as px
import numpy as np
import statsmodels.api as sm

Declare constants

In [50]:
# Month of ChatGPT launch
T_0 = pd.Timestamp("2022-11-01")
T_START = pd.Timestamp("2021-01-01")

Load long form dataframe and extract key values for analysis:
- Most recent month with data for all industries

In [40]:
df = pd.read_csv("../data/processed/employment.csv", parse_dates=["date"])
industry_codes = pd.read_csv("../data/reference/industry_code_map.csv")

# Get earliest and latest date for all industries in sample
t_start = df["date"].min()
t_end = df.groupby("industry_name")["date"].max().min()

# Number of unique industries in sample
unique_industries = df.naics.nunique()

df.head()

,date,industry_name,industry_code,naics,ai_exposure,employment,sector
0,2026-06-01,Offices of physicians,65621100,6211,1.009907,3053.7,Private education and health services
1,2026-05-01,Offices of physicians,65621100,6211,1.009907,3054.6,Private education and health services
2,2026-04-01,Offices of physicians,65621100,6211,1.009907,3050.7,Private education and health services
3,2026-03-01,Offices of physicians,65621100,6211,1.009907,3049.8,Private education and health services
4,2026-02-01,Offices of physicians,65621100,6211,1.009907,3011.8,Private education and health services


## Data coverage
As I was unable to match a significant number of AIIE scores to BLS series, I check the coverage of my dataset against the expanded out list of AIIE scores.

In [25]:
# Extract the number of covered industries and total employment for each sector
covered = (df[df["date"] == t_end]
    .groupby("sector")
    .agg(industries=("industry_name", "size"),
    employment=("employment", "sum")
    )
)

# Calculate total number of series in each sector


# Get unique 3-digit aggregate series IDs
agg_codes = df.loc[df["industry_code"].astype(str).str[5] == "0", "industry_code"].unique().tolist()
agg_codes

prefixes = [str(x)[:-3] for x in agg_codes]

prefixes

industry_codes = industry_codes.assign(
    digit_3 = industry_codes["industry_code"].astype(str).str[:-3],
    )
industry_codes.loc[industry_codes["3_dig"].isin(prefixes)]

KeyError: '3_dig'

In [ ]:
ss_df =  pd.json_normalize(
    ss_json["Results"]["series"], 
    record_path="data",
    meta="seriesID")

ss_df = (ss_df
    .assign(
        year=ss_df.year.astype(int),
        month=ss_df.period.str[1:].astype(int),
        industry=ss_df.seriesID.str[3:-2])
    .query(f"year == {t_end.year} & month == {t_end.month}")
)

ss_df = ss_df[["industry", "value"]]
# aiie = pd.read_csv("../data/reference/aiie.csv")
# aiie["naics_2"] = aiie["naics"].astype(str).str[:2]

NameError: name 'ss_json' is not defined

In [ ]:
sector_growth = (
    df.groupby("sector")
      .agg(
          emp_start=("emp_start","sum"),
          emp_end=("emp_end","sum")
      )
      .assign(
          growth_pct=lambda x:
          (x.emp_end / x.emp_start - 1) * 100
      )
)

fig = px.bar(
    df,
    x="sector",
    y="growth_pct",
    color="sector",
    title="Distribution of AI Exposure Scores by Sector",
    labels={
        "sector": "Sector",
        "ai_exposure": "AI Exposure Score (AIIE)"
    }
)

fig.update_layout(
    xaxis_title="Sector",
    yaxis_title="AI Exposure Score",
    showlegend=False,
    xaxis_tickangle=-45,
    template="simple_white",
    height=600
)

fig.show()

KeyError: "Label(s) ['emp_end', 'emp_start'] do not exist"

## Employment growth vs AI exposure
Create plot_df for each chart I want to use for analysis. Merge required tables from NB02

In [ ]:
first = (
    df.query(f"date == @T_0") # @ used to bring variable into query string
    .groupby("industry_name")
    .first()[["employment", "ai_exposure", "sector"]]
    .rename(columns={"employment": "emp_start"})
    )

last = (
    df.query(f"date == @t_end") 
    .groupby("industry_name")
    .first()[["employment", "ai_exposure", "sector"]]
    .rename(columns={"employment": "emp_end"})
    )

plot_df = (
    pd.merge(
        left=first, 
        right=last, 
        on=["industry_name", "sector", "ai_exposure"], 
        how="outer")
        .assign(
            emp_growth_pct=lambda x:
            (x.emp_end - x.emp_start) / x.emp_start *100
            )
        .reset_index()
)

fig = px.scatter(
    plot_df,
    x="ai_exposure",
    y="emp_growth_pct",
    color="sector",
    hover_name="industry_name",
    hover_data={
        "sector": False,      
        "emp_start": False,
        "ai_exposure": ":.2f",
        "emp_growth_pct": ":.1f",
    },
    labels={
        "ai_exposure": "AIIE score",
        "emp_growth_pct": "Change in employment (%)"
    },
    size="emp_start",
    trendline="ols",
    trendline_scope="overall", # Stop plotly from adding a trendline per sector
    title="There is a positive but weak correlation between AI exposure and employment growth"
)

fig.show()

# Have commented this out as it does not work in Nuvolos
# fig_path = "../docs/assets/emp_growth_ai.png"
# # fig.write_image(fig_path) 
# print(f"Saved figure to {fig_path}")

In [41]:
sector_summary = (
    df.groupby("sector")
      .apply(
          lambda g: pd.Series({
              "employment_growth_pct":
                  (g["emp_end"].sum() / g["emp_start"].sum() - 1) * 100,
              "avg_ai_exposure":
                  np.average(
                      g["ai_exposure"],
                      weights=g["emp_start"]
                  ),
              "employment":
                  g["emp_start"].sum()
          })
      )
      .reset_index()
)

fig = px.scatter(
    sector_summary,
    x="avg_ai_exposure",
    y="employment_growth_pct",
    size="employment",
    colour="sector",
    text="sector",
    title="Sector Employment Growth vs AI Exposure"
)

fig.update_traces(textposition="top center")

fig.update_layout(
    template="simple_white",
    xaxis_title="Average AI Exposure",
    yaxis_title="Employment Growth (%)",
    height=700
)

fig.show()

KeyError: 'emp_end'

In [71]:
baseline = (
    df.query(f"date == @T_0") # @ used to bring variable into query string
    [["industry_name", "employment", "ai_exposure", "sector"]]
    .rename(columns={"employment": "emp_start"})
    )

plot_df = (
    pd.merge(
        left=df.query("@T_0 <= date <= @t_end"), 
        right=baseline, 
        on=["industry_name", "sector", "ai_exposure"], 
        how="inner")
        .assign(
            # Cumulative employment change from ChatGPT launch
            emp_growth_pct=lambda x:
            (x.employment - x.emp_start) / x.emp_start *100,
            month=lambda x: x.date.dt.strftime("%Y-%m")
            )
        .sort_values(["date", "industry_name"])
        .reset_index()
)

fig = px.scatter(
    plot_df,
    x="ai_exposure",
    y="emp_growth_pct",
    color="sector",
    size="emp_start",
    opacity=0.7,
    hover_name="industry_name",
    hover_data={
        "sector": False,      
        "emp_start": False,
        "ai_exposure": ":.2f",
        "emp_growth_pct": ":.1f",
    },
    labels={
        "ai_exposure": "AIIE score",
        "emp_growth_pct": "Change in employment (%)",
        "sector": "Sector",
    },
    animation_frame="month",
    animation_group="industry_name",
    range_y=[-25, 35],
    title=(
        "<b>There is a positive but weak correlation between"
        "<br>AI exposure and employment growth</b>"
        "<br><sup>Employment growth by industry relative to AI industry exposure scores</sup>"

    ),
)

# Standalone trendline for whole sample
fig_ols = px.scatter(
    plot_df,
    x="ai_exposure",
    y="emp_growth_pct",
    animation_frame="month",
    trendline="ols",
    trendline_color_override="grey"
)

def get_line(traces):
    """px gives trendlines mode='lines'; the scatter points are 'markers'."""
    line = next(t for t in traces if t.mode == "lines")
    line.update(name="Trend (OLS)", showlegend=True, legendgroup="ols")
    return line

# Line for the frame shown before Play is pressed
fig.add_traces(get_line(fig_ols.data))

# Same line appended to every frame
ols_by_frame = {f.name: get_line(f.data) for f in fig_ols.frames}

for frame in fig.frames:
    frame.data = frame.data + (ols_by_frame[frame.name],)
    frame.traces = list(range(len(frame.data)))

fig.show()

fig_path = "../docs/assets/emp_growth_ai_anim.html"
fig.write_html(fig_path)
print(f"Saved figure to {fig_path}")

/opt/conda/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Saved figure to ../docs/assets/emp_growth_ai_anim.html


### How did the relationship change over time?

The above chart shows a steepening of the regression line over time following the launch of ChatGPT.

In [53]:
baseline = (
    df.query(f"date == @T_START")
    [["industry_name", "employment", "ai_exposure", "sector"]]
    .rename(columns={"employment": "emp_start"})
    )

plot_df = (
    pd.merge(
        left=df.query("@T_START <= date <= @t_end"), 
        right=baseline, 
        on=["industry_name", "sector", "ai_exposure"], 
        how="inner")
        .assign(
            # Cumulative employment change from ChatGPT launch
            emp_growth_pct=lambda x:
            (x.employment - x.emp_start) / x.emp_start *100,
            month=lambda x: x.date.dt.strftime("%Y-%m")
            )
        .sort_values(["date", "industry_name"])
        .reset_index()
)

coefs = []

for date, d in plot_df.groupby("date"):
    model = sm.OLS( d["emp_growth_pct"], 
    sm.add_constant(d["ai_exposure"]) 
    ).fit()

    coefs.append({
        "date": date,
        "beta": model.params["ai_exposure"],
        "lower": model.conf_int().loc["ai_exposure", 0],
        "upper": model.conf_int().loc["ai_exposure", 1],
        "r2": model.rsquared,}
        )

    coef_df = pd.DataFrame(coefs).sort_values("date")

/opt/conda/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


In [65]:
fig = px.line(
    coef_df,
    x="date",
    y="beta",
    labels={
        "date": "",
        "beta": "Coefficient on AI exposure",
        },
    title=(
        "<b>The correlation between AI exposure and employment growth" 
        "<br>emerged after the launch of ChatGPT</b>"
        "<br><sup>Monthly OLS coefficient with 95% confidence intervals</sup>"
    )
)

fig.add_scatter(
    x=coef_df["date"],
    y=coef_df["upper"],
    mode="lines",
    line=dict(width=0),
    showlegend=False
    )

fig.add_scatter(
    x=coef_df["date"],
    y=coef_df["lower"],
    mode="lines",
    fill="tonexty",
    fillcolor="rgba(0,100,255,0.15)",
    line=dict(width=0),
    name="95% CI"
    )

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="grey"
    )

fig.add_vline(
    x=T_0,
    line_dash="dot",
    annotation_text="ChatGPT launch"
)
fig.update_layout(showlegend=False)
fig.show()

## Sensitivity analysis
With and without 44 and 45 codes: 